In [ ]:
import chromadb

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [ ]:
# 1. EL DOCUMENTO
texto = """
El esquema de vacunación para perros de raza mediana inicia entre las 6 y 8 semanas de vida.
Las vacunas esenciales incluyen Parvovirus, Moquillo, Hepatitis Infecciosa, Leptospirosis y Rabia.
Se recomienda la vacuna contra la Tos de las Perreras (Bordetella) para perros con alta socialización en parques.
Es obligatorio realizar una desparasitación interna y externa antes de aplicar cualquier refuerzo inmunológico.
Los perros adultos requieren un refuerzo anual de la vacuna polivalente y la rabia para mantener la inmunidad.
"""

In [ ]:
# 2. VISUALIZANDO EL CHUNKING
print("--- PASO 1: CHUNKING ---")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10)
chunks = text_splitter.split_text(texto)
for i, chunk in enumerate(chunks):
    print(f"Pedazo {i}: [{chunk}]")
time.sleep(1)

In [ ]:
# 3. VISUALIZANDO EL EMBEDDING
print("\n--- PASO 2: EMBEDDINGS (Vectores) ---")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
ejemplo_vector = model.encode(chunks[0])
print(f"Texto: '{chunks[0]}'")
print(f"Se convirtió en un vector de {len(ejemplo_vector)} números.")
print(f"Primeros 5 números del vector: {ejemplo_vector[:5]} ...")

In [ ]:
# 4. GUARDANDO EN CHROMADB
print("\n--- PASO 3: GUARDANDO EN LA BASE DE DATOS ---")
db_path = "data/chroma_db"

client = chromadb.PersistentClient(path=db_path)
collection = client.get_or_create_collection(name="vacunas_mascotas")
for i, chunk in enumerate(chunks):
    vector = model.encode(chunk).tolist()
    collection.add(ids=[f"id_{i}"], embeddings=[vector], documents=[chunk])
print("¡Todo guardado y listo para buscar!")

In [ ]:
# 5. BUSCANDO Y VIENDO RESULTADOS
print("\n--- PASO 4: BÚSQUEDA ---")
query = "¿Qué tipo de vacuna sería la adecuada para un perro de raza mediana?"
print(f"Pregunta del usuario: '{query}'")

In [ ]:
query_vec = model.encode(query).tolist()
resultados = collection.query(query_embeddings=[query_vec], n_results=2)

In [ ]:
print("\nResultados encontrados por orden de relevancia:")
for i in range(len(resultados['documents'][0])):
    doc = resultados['documents'][0][i]
    distancia = resultados['distances'][0][i]
    print(f"Ranking {i+1}: {doc} (Distancia: {distancia:.4f})")

In [ ]:
col = client.get_collection("seguridad")
docs = col.get(include=["documents", "embeddings"])

for i, (doc, emb) in enumerate(zip(docs["documents"], docs["embeddings"]), start=1):
    print(f"Documento {i}:")
    print(f"Vector: {emb[:5]} ...")
    print(doc)
    print()
